# 01 — Data Loading & Cleaning

## Dataset
UCI Online Retail II — 1M+ transactions from a UK-based online retailer (2009–2011).  
Source: https://archive.ics.uci.edu/dataset/502/online+retail+ii

## What this notebook does
- Loads and combines both year sheets into a single dataframe
- Fixes column names, data types, and date formats
- Removes cancelled orders, negative quantities, and zero prices
- Drops rows with no Customer ID (required for RFM segmentation)
- Removes internal non-product stock codes
- Engineers the `revenue` column (quantity × price)
- Saves a clean CSV ready for analysis

## Key numbers after cleaning
| Metric | Value |
|---|---|
| Raw rows | ~1,067,000 |
| Clean rows | ~803,000 |
| Unique customers | ~5,900 |
| Unique products | ~4,600 |
| Date range | Dec 2009 → Dec 2011 |

Imports

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


Load both sheets and combine them

In [47]:
file_path = '../data/raw/online_retail_II.xlsx'

df_09 = pd.read_excel(file_path, sheet_name='Year 2009-2010', engine='openpyxl')
df_10 = pd.read_excel(file_path, sheet_name='Year 2010-2011', engine='openpyxl')

df = pd.concat([df_09, df_10], ignore_index=True)

print(f"Total rows loaded: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
df.head(100)

Total rows loaded: 1,067,371
Total columns: 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
95,489441,22111,SCOTTIE DOG HOT WATER BOTTLE,48,2009-12-01 09:44:00,4.25,18087.0,United Kingdom
96,489442,21955,UNION JACK GUNS & ROSES DOORMAT,2,2009-12-01 09:46:00,6.75,13635.0,United Kingdom
97,489442,22111,SCOTTIE DOG HOT WATER BOTTLE,3,2009-12-01 09:46:00,4.95,13635.0,United Kingdom
98,489442,22296,HEART IVORY TRELLIS LARGE,12,2009-12-01 09:46:00,1.65,13635.0,United Kingdom


First look at the data

In [48]:
print("=== Shape ===")
print(df.shape)

print("\n=== Column types ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Sample rows ===")
df.sample(5)

=== Shape ===
(1067371, 8)

=== Column types ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

=== Missing values ===
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

=== Sample rows ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
931510,571728,21623,VINTAGE UNION JACK MEMOBOARD,2,2011-10-19 09:18:00,9.95,16123.0,United Kingdom
176164,506137,22326,ROUND SNACK BOXES SET OF4 WOODLAND,6,2010-04-27 15:31:00,2.95,13319.0,United Kingdom
107535,499687,79323W,WHITE CHERRY LIGHTS,4,2010-03-02 10:16:00,6.75,13468.0,United Kingdom
400571,527754,21032,SPACE CADET WHITE,3,2010-10-18 17:05:00,1.25,15710.0,United Kingdom
177756,506250,21328,BALLOONS WRITING SET,1,2010-04-28 13:34:00,1.65,16327.0,United Kingdom


Rename columns for ease of use

In [49]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['country'] = df['country'].replace({'EIRE': 'Ireland'})

print("Renamed columns:", df.columns.tolist())

Renamed columns: ['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


Fix data types

In [50]:
# InvoiceDate should be datetime
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

# Customer ID should be string (it's an ID, not a number to calculate with)
df['customer_id'] = df['customer_id'].astype('Int64').astype('str')
df['customer_id'] = df['customer_id'].replace('<NA>', np.nan)

print("Date range:", df['invoicedate'].min(), "→", df['invoicedate'].max())
print("Types fixed ✓")

Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Types fixed ✓


Remove cancelled orders

In [51]:
print(f"Rows before removing cancellations: {len(df):,}")

# Cancelled invoices start with 'C'
cancelled = df[df['invoice'].astype(str).str.startswith('C')]
print(f"Cancelled transactions found: {len(cancelled):,}")

df = df[~df['invoice'].astype(str).str.startswith('C')]

# Also remove any remaining negative quantities
df = df[df['quantity'] > 0]

# And zero or negative prices
df = df[df['price'] > 0]

print(f"Rows after removing cancellations: {len(df):,}")

Rows before removing cancellations: 1,067,371
Cancelled transactions found: 19,494
Rows after removing cancellations: 1,041,670


Handle missing Customer IDs

In [52]:
print(f"Rows with missing Customer ID: {df['customer_id'].isnull().sum():,}")

# We drop them for the RFM analysis — we can't segment anonymous customers
df_clean = df.dropna(subset=['customer_id']).copy()

print(f"Rows after dropping null Customer IDs: {len(df_clean):,}")
print(f"Rows dropped: {len(df) - len(df_clean):,}")

Rows with missing Customer ID: 236,121
Rows after dropping null Customer IDs: 805,549
Rows dropped: 236,121


Remove bad stock codes

In [53]:
# These are non-product codes to exclude
bad_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK']

df_clean = df_clean[~df_clean['stockcode'].astype(str).str.upper().isin(bad_codes)]

print(f"Rows after removing non-product codes: {len(df_clean):,}")

Rows after removing non-product codes: 802,932


Create the revenue column

In [54]:
df_clean['revenue'] = df_clean['quantity'] * df_clean['price']

print(f"Total revenue in dataset: £{df_clean['revenue'].sum():,.2f}")
print(f"Average order value: £{df_clean.groupby('invoice')['revenue'].sum().mean():,.2f}")

Total revenue in dataset: £17,451,756.30
Average order value: £476.24


Final quality check

In [55]:
print("=== Final dataset summary ===")
print(f"Rows: {len(df_clean):,}")
print(f"Unique customers: {df_clean['customer_id'].nunique():,}")
print(f"Unique invoices: {df_clean['invoice'].nunique():,}")
print(f"Unique products: {df_clean['stockcode'].nunique():,}")
print(f"Countries: {df_clean['country'].nunique()}")
print(f"Date range: {df_clean['invoicedate'].min().date()} → {df_clean['invoicedate'].max().date()}")
print(f"\nMissing values:\n{df_clean.isnull().sum()}")

=== Final dataset summary ===
Rows: 802,932
Unique customers: 5,862
Unique invoices: 36,645
Unique products: 4,625
Countries: 41
Date range: 2009-12-01 → 2011-12-09

Missing values:
invoice        0
stockcode      0
description    0
quantity       0
invoicedate    0
price          0
customer_id    0
country        0
revenue        0
dtype: int64


Save the clean dataset

In [56]:
df_clean.to_csv('../data/online_retail_clean.csv', index=False)
print("Clean dataset saved ✓")

Clean dataset saved ✓


In [57]:
df_clean.head()

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [58]:
df_clean.sample(5)

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
907379,569897,22138,BAKING SET 9 PIECE RETROSPOT,1,2011-10-06 16:01:00,4.95,17813,United Kingdom,4.95
946124,572902,22952,60 CAKE CASES VINTAGE CHRISTMAS,288,2011-10-26 15:01:00,0.55,14866,United Kingdom,158.40
450163,532143,22381,TOY TIDY PINK POLKADOT,1,2010-11-11 11:44:00,2.10,16782,United Kingdom,2.10
887032,568346,85129D,BEADED CRYSTAL HEART PINK SMALL,1,2011-09-26 15:28:00,2.46,14096,United Kingdom,2.46
202837,508964,20675,BLUE SPOTTY BOWL,16,2010-05-19 11:37:00,1.25,15738,United Kingdom,20.00
